In [1]:
from kaggle_secrets import UserSecretsClient
import wandb

# Initialize the client to access secrets
user_secrets = UserSecretsClient()

# Get the key you just stored
api_key = user_secrets.get_secret("WANDB_API_KEY") 

# Log in to wandb
wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saurabh200206 (saurabh200206-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses, models
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
import numpy as np


2025-09-11 19:00:48.380512: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757617248.703952      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757617248.795286      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
print("Loading and preparing data...")
train_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")

# Create the same input text as in Phase 1
train_df['input_text'] = train_df.apply(
    lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", 
    axis=1
)

Loading and preparing data...


In [4]:
train_df['Category'] = train_df['Category'].astype(str)
train_df['Misconception'] = train_df['Misconception'].astype(str)
train_df['full_label'] = train_df['Category'] + ':' + train_df['Misconception']
train_df['label_id'] = pd.Categorical(train_df['full_label']).codes

In [5]:
train_examples = []
for i in range(len(train_df) - 1):
    text1 = train_df.iloc[i]['input_text']
    text2 = train_df.iloc[i + 1]['input_text']
    label = 1 if train_df.iloc[i]['label_id'] == train_df.iloc[i + 1]['label_id'] else 0
    train_examples.append(InputExample(texts=[text1, text2], label=label))


In [6]:
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
embedding_model = models.Transformer(MODEL_NAME, max_seq_length=512)
pooling_model = models.Pooling(embedding_model.get_word_embedding_dimension())
model = SentenceTransformer(modules=[embedding_model, pooling_model])


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [7]:
train_loss = losses.OnlineContrastiveLoss(model=model, margin=0.5)


In [8]:
print("Starting training...")
# The DataLoader will handle batching the InputExamples.
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16,drop_last=True)

# We'll train for one epoch. Fine-tuning embedding models is often fast.
num_epochs = 1
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% of steps for warmup

model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=num_epochs,
          warmup_steps=warmup_steps,
          output_path='./sbert-finetuned-model', # Save the model here
          show_progress_bar=True)

print("Phase 2 training complete. Model saved to './sbert-finetuned-model'.")


Starting training...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.
wandb: Tracking run with wandb version 0.20.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20250911_190120-uvthxuwo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run checkpoints/model
wandb: ⭐️ View project at https://wandb.ai/saurabh200206-self/sentence-transformers
wandb: 🚀 View run at https://wandb.ai/saurabh200206-self/sentence-transformers/runs/uvthxuwo


Step,Training Loss
500,1.487600
1000,1.299800


Phase 2 training complete. Model saved to './sbert-finetuned-model'.
